<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/01_Raw_Dataset_Loading_%26_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ==============================================================================
# SPP-GAN RESEARCH PROJECT
# NOTEBOOK 01 — RAW DATASET LOADING & VALIDATION
# ==============================================================================
#
# Purpose:
#   Acquire, preserve, load, and audit the official raw datasets used by the
#   SPP-GAN research framework.
#
# Scientific scope:
#   - Official raw dataset acquisition
#   - Source-file preservation
#   - Raw structural validation
#   - Target-column identification
#   - Raw missing-value characterization
#   - Duplicate-record detection
#   - Constant / zero-variance feature detection
#   - Identifier-candidate detection
#   - Numeric / categorical inventory
#   - Raw-file SHA-256 fingerprinting
#   - Dataset-level validation
#   - Machine-readable artifact persistence
#   - Reproducibility / provenance recording
#
# Explicitly NOT performed here:
#   - Imputation
#   - Encoding
#   - Scaling / normalization
#   - Feature engineering
#   - Feature selection
#   - Duplicate removal
#   - Identifier removal
#   - Train/validation/test splitting
#   - Model training
#   - Synthetic-data generation
#   - Differential privacy
#   - Statistical fidelity evaluation
#   - ML utility evaluation
#
# Notebook 00 is the single source of truth for global configuration.
#
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import hashlib
import platform
import re
import shutil
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("SPP-GAN RESEARCH PROJECT")
print("NOTEBOOK 01 — RAW DATASET LOADING & VALIDATION")
print("=" * 100)
print()
print("Scope:")
print("  Official raw-data acquisition, preservation, loading, and validation.")
print()
print("Scientific transformations:")
print("  NONE")
print()
print("Notebook 00 configuration:")
print("  REQUIRED")
print("=" * 100)

SPP-GAN RESEARCH PROJECT
NOTEBOOK 01 — RAW DATASET LOADING & VALIDATION

Scope:
  Official raw-data acquisition, preservation, loading, and validation.

Scientific transformations:
  NONE

Notebook 00 configuration:
  REQUIRED


In [16]:
# ==============================================================================
# 01.1 — IMPORT CORE LIBRARIES
# ==============================================================================

import os
import sys
import json
import hashlib
import shutil
import zipfile
import tarfile
import time
import platform
import re
import warnings

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("01.1 — IMPORT CORE LIBRARIES")
print("=" * 100)

print(f"✓ Python : {sys.version.split()[0]}")
print(f"✓ pandas : {pd.__version__}")
print(f"✓ numpy  : {np.__version__}")
print("✓ Core libraries imported successfully.")

01.1 — IMPORT CORE LIBRARIES
✓ Python : 3.13.15
✓ pandas : 2.2.3
✓ numpy  : 2.1.3
✓ Core libraries imported successfully.


In [17]:
# ==============================================================================
# 01.2 — GOOGLE DRIVE + NOTEBOOK 00 CONFIGURATION
# ==============================================================================

print("=" * 100)
print("01.2 — GOOGLE DRIVE + NOTEBOOK 00 CONFIGURATION")
print("=" * 100)

# ------------------------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------------------------

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

    if not Path("/content/drive/MyDrive").exists():
        raise RuntimeError("Google Drive mounted but MyDrive is unavailable.")

    print("✓ Google Drive is available.")

except ImportError:
    print("⚠ Google Colab environment not detected.")
    print("  Continuing only if the configured project root is accessible.")

# ------------------------------------------------------------------------------
# Project root
# ------------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"SPP-GAN project root does not exist:\n{PROJECT_ROOT}\n"
        "Run Notebook 00 first."
    )

print(f"✓ Project root : {PROJECT_ROOT}")

# ------------------------------------------------------------------------------
# Notebook 00 configuration directory
# ------------------------------------------------------------------------------

NOTEBOOK_00_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_00"
)

CONFIG_DIR = NOTEBOOK_00_DIR / "config"

if not CONFIG_DIR.exists():
    raise FileNotFoundError(
        f"Notebook 00 configuration directory not found:\n{CONFIG_DIR}\n"
        "Run Notebook 00 completely before Notebook 01."
    )

# ------------------------------------------------------------------------------
# Required Notebook 00 configuration artifacts
# ------------------------------------------------------------------------------

REQUIRED_CONFIG_FILES = {
    "experiment": CONFIG_DIR / "experiment_config.json",
    "dataset_registry": CONFIG_DIR / "dataset_registry.json",
    "model_registry": CONFIG_DIR / "model_registry.json",
    "evaluation": CONFIG_DIR / "evaluation_config.json",
    "privacy": CONFIG_DIR / "privacy_config.json",
    "sppgan": CONFIG_DIR / "sppgan_config.json",
}

for name, path in REQUIRED_CONFIG_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Required Notebook 00 configuration missing: {path}"
        )

# ------------------------------------------------------------------------------
# Load configurations
# ------------------------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

EXPERIMENT_CONFIG = load_json(REQUIRED_CONFIG_FILES["experiment"])
DATASET_REGISTRY = load_json(REQUIRED_CONFIG_FILES["dataset_registry"])
MODEL_REGISTRY = load_json(REQUIRED_CONFIG_FILES["model_registry"])
EVALUATION_CONFIG = load_json(REQUIRED_CONFIG_FILES["evaluation"])
PRIVACY_CONFIG = load_json(REQUIRED_CONFIG_FILES["privacy"])
SPPGAN_CONFIG = load_json(REQUIRED_CONFIG_FILES["sppgan"])

print("✓ experiment_config.json loaded")
print("✓ dataset_registry.json loaded")
print("✓ model_registry.json loaded")
print("✓ evaluation_config.json loaded")
print("✓ privacy_config.json loaded")
print("✓ sppgan_config.json loaded")

print()
print("Notebook 00 configuration successfully loaded.")

01.2 — GOOGLE DRIVE + NOTEBOOK 00 CONFIGURATION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive is available.
✓ Project root : /content/drive/MyDrive/SPP_GAN_Research
✓ experiment_config.json loaded
✓ dataset_registry.json loaded
✓ model_registry.json loaded
✓ evaluation_config.json loaded
✓ privacy_config.json loaded
✓ sppgan_config.json loaded

Notebook 00 configuration successfully loaded.


In [18]:
# ==============================================================================
# 01.3 — VERIFY PROJECT STRUCTURE
# ==============================================================================

print("=" * 100)
print("01.3 — VERIFY PROJECT STRUCTURE")
print("=" * 100)

REQUIRED_DIRS = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "results",
    PROJECT_ROOT / "results" / "notebooks",
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "publication",
    PROJECT_ROOT / "manifests",
    PROJECT_ROOT / "configs",
    PROJECT_ROOT / "environment",
    PROJECT_ROOT / "logs",
    PROJECT_ROOT / "tmp",
]

for directory in REQUIRED_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

    if not directory.exists():
        raise RuntimeError(f"Unable to create/access directory: {directory}")

    print(f"✓ {directory}")

print()
print("✓ Project structure verified.")

01.3 — VERIFY PROJECT STRUCTURE
✓ /content/drive/MyDrive/SPP_GAN_Research/data
✓ /content/drive/MyDrive/SPP_GAN_Research/data/raw
✓ /content/drive/MyDrive/SPP_GAN_Research/data/processed
✓ /content/drive/MyDrive/SPP_GAN_Research/models
✓ /content/drive/MyDrive/SPP_GAN_Research/results
✓ /content/drive/MyDrive/SPP_GAN_Research/results/notebooks
✓ /content/drive/MyDrive/SPP_GAN_Research/reports
✓ /content/drive/MyDrive/SPP_GAN_Research/publication
✓ /content/drive/MyDrive/SPP_GAN_Research/manifests
✓ /content/drive/MyDrive/SPP_GAN_Research/configs
✓ /content/drive/MyDrive/SPP_GAN_Research/environment
✓ /content/drive/MyDrive/SPP_GAN_Research/logs
✓ /content/drive/MyDrive/SPP_GAN_Research/tmp

✓ Project structure verified.


In [19]:
# ==============================================================================
# 01.4 — RESOLVE DATASET IDS
# ==============================================================================

print("=" * 100)
print("01.4 — RESOLVE DATASET IDS")
print("=" * 100)

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

registered_ids = set(DATASET_REGISTRY.keys())

missing_datasets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in registered_ids
]

if missing_datasets:
    raise RuntimeError(
        "The following required datasets are missing from Notebook 00 registry:\n"
        + "\n".join(f"  - {x}" for x in missing_datasets)
    )

DATASET_IDS = EXPECTED_DATASETS.copy()

print(f"✓ Registered datasets : {len(DATASET_IDS)}")

for dataset_id in DATASET_IDS:
    print(f"  ✓ {dataset_id}")

if len(DATASET_IDS) != 3:
    raise RuntimeError(
        f"Expected exactly 3 research datasets; found {len(DATASET_IDS)}."
    )

print()
print("✓ Dataset registry integrity verified.")

01.4 — RESOLVE DATASET IDS
✓ Registered datasets : 3
  ✓ adult_income
  ✓ bank_marketing
  ✓ diabetes_130us

✓ Dataset registry integrity verified.


In [20]:
# ==============================================================================
# 01.5 — OFFICIAL RAW DATASET ACQUISITION & PRESERVATION
# ==============================================================================

print("=" * 100)
print("01.5 — OFFICIAL RAW DATASET ACQUISITION & PRESERVATION")
print("=" * 100)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DOWNLOAD_DIR = PROJECT_ROOT / "tmp" / "raw_dataset_downloads"
NOTEBOOK_01_DIR = PROJECT_ROOT / "results" / "notebooks" / "notebook_01"
MANIFEST_DIR = NOTEBOOK_01_DIR / "manifest"

RAW_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Raw data directory : {RAW_DIR}")
print(f"✓ Download directory : {DOWNLOAD_DIR}")
print(f"✓ Manifest directory : {MANIFEST_DIR}")

# ------------------------------------------------------------------------------
# Official UCI URLs
# ------------------------------------------------------------------------------

UCI_URLS = {
    "adult_income": (
        "https://archive.ics.uci.edu/static/public/2/adult.zip"
    ),
    "bank_marketing": (
        "https://archive.ics.uci.edu/static/public/222/"
        "bank%2Bmarketing.zip"
    ),
    "diabetes_130us": (
        "https://archive.ics.uci.edu/static/public/296/"
        "diabetes%2B130-us%2Bhospitals%2Bfor%2Byears%2B1999-2008.zip"
    ),
}

RAW_OUTPUT_FILES = {
    "adult_income": RAW_DIR / "adult_income.csv",
    "bank_marketing": RAW_DIR / "bank_marketing.csv",
    "diabetes_130us": RAW_DIR / "diabetes_130us.csv",
}

ORIGINAL_DIRS = {
    dataset_id: RAW_DIR / f"{dataset_id}_original"
    for dataset_id in DATASET_IDS
}

DOWNLOAD_FILES = {
    dataset_id: DOWNLOAD_DIR / f"{dataset_id}.zip"
    for dataset_id in DATASET_IDS
}

# ------------------------------------------------------------------------------
# SHA-256
# ------------------------------------------------------------------------------

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)

    return sha256.hexdigest()

# ------------------------------------------------------------------------------
# Download helper
# ------------------------------------------------------------------------------

def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size > 0:
        print(f"  ✓ Archive already exists: {destination}")
        return

    print(f"  Downloading:")
    print(f"    URL : {url}")

    try:
        import requests
    except ImportError:
        raise RuntimeError("requests package is required for dataset download.")

    response = requests.get(
        url,
        stream=True,
        timeout=120,
        headers={"User-Agent": "SPP-GAN-Research/1.0"},
    )

    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))
    downloaded = 0

    with open(destination, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

    if not destination.exists() or destination.stat().st_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {destination}")

    if total:
        print(
            f"  ✓ Downloaded {downloaded / (1024**2):.2f} MB "
            f"of {total / (1024**2):.2f} MB"
        )
    else:
        print(f"  ✓ Downloaded {downloaded / (1024**2):.2f} MB")

# ------------------------------------------------------------------------------
# Safe ZIP extraction
# ------------------------------------------------------------------------------

def safe_extract_zip(zip_path, extract_dir):
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:

        base = extract_dir.resolve()

        for member in zf.infolist():
            target = (extract_dir / member.filename).resolve()

            if not str(target).startswith(str(base)):
                raise RuntimeError(
                    f"Unsafe ZIP member detected: {member.filename}"
                )

        zf.extractall(extract_dir)

# ------------------------------------------------------------------------------
# Nested ZIP extraction
# ------------------------------------------------------------------------------

def extract_nested_zip_archives(root_dir, max_levels=5):
    """
    Recursively extracts ZIP files found inside extracted directories.

    This is required because the official UCI Bank Marketing download contains
    bank.zip, which itself contains bank-full.csv.
    """

    root_dir = Path(root_dir)

    processed = set()

    for level in range(max_levels):

        zip_files = [
            p for p in root_dir.rglob("*.zip")
            if p.is_file()
        ]

        new_zip_files = [
            p for p in zip_files
            if str(p.resolve()) not in processed
        ]

        if not new_zip_files:
            break

        print(
            f"  Nested extraction level {level + 1}: "
            f"{len(new_zip_files)} ZIP archive(s)"
        )

        for zip_file in new_zip_files:

            processed.add(str(zip_file.resolve()))

            extract_dir = zip_file.parent / (
                zip_file.stem + "_extracted"
            )

            extract_dir.mkdir(parents=True, exist_ok=True)

            try:
                safe_extract_zip(zip_file, extract_dir)
                print(f"    ✓ {zip_file.name}")

            except Exception as exc:
                raise RuntimeError(
                    f"Failed to extract nested archive:\n"
                    f"{zip_file}"
                ) from exc

# ------------------------------------------------------------------------------
# Filename normalization
# ------------------------------------------------------------------------------

def normalize_filename(name):
    return re.sub(
        r"[^a-z0-9]",
        "",
        Path(name).name.lower()
    )

def find_exact_normalized_files(root_dir, expected_names):
    root_dir = Path(root_dir)

    expected_normalized = {
        normalize_filename(name): name
        for name in expected_names
    }

    matches = {
        expected_name: []
        for expected_name in expected_names
    }

    for path in root_dir.rglob("*"):
        if not path.is_file():
            continue

        normalized = normalize_filename(path.name)

        if normalized in expected_normalized:
            expected_name = expected_normalized[normalized]
            matches[expected_name].append(path)

    return matches

# ------------------------------------------------------------------------------
# Acquisition manifest
# ------------------------------------------------------------------------------

ACQUISITION_MANIFEST = {
    "notebook": "01",
    "purpose": "Official raw dataset acquisition and preservation",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source": "UCI Machine Learning Repository",
    "datasets": {},
}

# ------------------------------------------------------------------------------
# Process datasets
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"DATASET: {dataset_id}")
    print("-" * 100)

    output_file = RAW_OUTPUT_FILES[dataset_id]
    original_dir = ORIGINAL_DIRS[dataset_id]
    download_file_path = DOWNLOAD_FILES[dataset_id]
    source_url = UCI_URLS[dataset_id]

    record = {
        "dataset_id": dataset_id,
        "source_url": source_url,
        "download_archive": str(download_file_path),
        "final_raw_file": str(output_file),
        "original_preservation_directory": str(original_dir),
        "archive_sha256": None,
        "raw_file_sha256": None,
        "status": "FAILED",
    }

    # --------------------------------------------------------------------------
    # Existing standardized dataset
    # --------------------------------------------------------------------------

    if output_file.exists() and output_file.stat().st_size > 0:

        print("✓ Final raw dataset already exists:")
        print(f"  {output_file}")

        record["raw_file_sha256"] = calculate_sha256(output_file)
        record["status"] = "EXISTING_VALID_FILE"

        ACQUISITION_MANIFEST["datasets"][dataset_id] = record

        continue

    # --------------------------------------------------------------------------
    # Download
    # --------------------------------------------------------------------------

    print("Downloading official UCI archive...")

    download_file(
        source_url,
        download_file_path
    )

    archive_hash = calculate_sha256(download_file_path)

    record["archive_sha256"] = archive_hash

    print(f"  Archive SHA-256:")
    print(f"    {archive_hash}")

    # --------------------------------------------------------------------------
    # Extraction
    # --------------------------------------------------------------------------

    extraction_dir = DOWNLOAD_DIR / f"{dataset_id}_extracted"

    if extraction_dir.exists():
        shutil.rmtree(extraction_dir)

    extraction_dir.mkdir(parents=True, exist_ok=True)

    print()
    print("Extracting archive...")

    safe_extract_zip(
        download_file_path,
        extraction_dir
    )

    # Critical correction:
    # recursively extract nested ZIP archives.
    extract_nested_zip_archives(
        extraction_dir,
        max_levels=5
    )

    print("✓ Archive extraction completed.")

    # --------------------------------------------------------------------------
    # Dataset-specific source resolution
    # --------------------------------------------------------------------------

    if dataset_id == "adult_income":

        matches = find_exact_normalized_files(
            extraction_dir,
            [
                "adult.data",
                "adult.test",
                "adult.names",
            ]
        )

        adult_data = matches["adult.data"]
        adult_test = matches["adult.test"]
        adult_names = matches["adult.names"]

        if not adult_data or not adult_test:
            raise FileNotFoundError(
                "Adult dataset source files adult.data and adult.test "
                "could not be located."
            )

        # Preserve original source files.
        original_dir.mkdir(parents=True, exist_ok=True)

        for source_file in adult_data + adult_test + adult_names:
            destination = original_dir / source_file.name

            if not destination.exists():
                shutil.copy2(source_file, destination)

        adult_columns = [
            "age",
            "workclass",
            "fnlwgt",
            "education",
            "education_num",
            "marital_status",
            "occupation",
            "relationship",
            "race",
            "sex",
            "capital_gain",
            "capital_loss",
            "hours_per_week",
            "native_country",
            "income",
        ]

        train_df = pd.read_csv(
            adult_data[0],
            header=None,
            names=adult_columns,
            skipinitialspace=True,
            dtype=str,
        )

        test_df = pd.read_csv(
            adult_test[0],
            header=None,
            names=adult_columns,
            skiprows=1,
            skipinitialspace=True,
            dtype=str,
        )

        # Source-format normalization only:
        # Adult test labels have a trailing period.
        test_df["income"] = (
            test_df["income"]
            .astype(str)
            .str.strip()
            .str.rstrip(".")
        )

        train_df["income"] = (
            train_df["income"]
            .astype(str)
            .str.strip()
        )

        adult_df = pd.concat(
            [train_df, test_df],
            axis=0,
            ignore_index=True
        )

        adult_df.to_csv(
            output_file,
            index=False
        )

        del train_df
        del test_df
        del adult_df

    elif dataset_id == "bank_marketing":

        print()
        print("Locating Bank Marketing source file...")

        matches = find_exact_normalized_files(
            extraction_dir,
            ["bank-full.csv"]
        )

        bank_matches = matches["bank-full.csv"]

        if len(bank_matches) == 0:

            csv_files = sorted(
                [
                    p for p in extraction_dir.rglob("*.csv")
                    if p.is_file()
                ]
            )

            print()
            print("CSV files discovered:")
            for p in csv_files:
                print(f"  - {p}")

            raise FileNotFoundError(
                "Bank Marketing full dataset could not be located.\n"
                "The official archive was downloaded and nested ZIP "
                "archives were extracted, but bank-full.csv was not found."
            )

        if len(bank_matches) > 1:

            print("Multiple bank-full.csv files detected:")

            for p in bank_matches:
                print(f"  - {p}")

            raise RuntimeError(
                "Ambiguous Bank Marketing source file resolution."
            )

        source_file = bank_matches[0]

        print(f"✓ Source file located:")
        print(f"  {source_file}")

        original_dir.mkdir(parents=True, exist_ok=True)

        preserved_file = original_dir / source_file.name

        if not preserved_file.exists():
            shutil.copy2(
                source_file,
                preserved_file
            )

        shutil.copy2(
            source_file,
            output_file
        )

    elif dataset_id == "diabetes_130us":

        print()
        print("Locating Diabetes source file...")

        matches = find_exact_normalized_files(
            extraction_dir,
            ["diabetic_data.csv"]
        )

        diabetes_matches = matches["diabetic_data.csv"]

        if len(diabetes_matches) == 0:

            csv_files = sorted(
                [
                    p for p in extraction_dir.rglob("*.csv")
                    if p.is_file()
                ]
            )

            print()
            print("CSV files discovered:")
            for p in csv_files:
                print(f"  - {p}")

            raise FileNotFoundError(
                "Diabetes 130-US Hospitals source file could not be located."
            )

        if len(diabetes_matches) > 1:

            print("Multiple diabetic_data.csv files detected:")

            for p in diabetes_matches:
                print(f"  - {p}")

            raise RuntimeError(
                "Ambiguous Diabetes source file resolution."
            )

        source_file = diabetes_matches[0]

        print(f"✓ Source file located:")
        print(f"  {source_file}")

        original_dir.mkdir(parents=True, exist_ok=True)

        preserved_file = original_dir / source_file.name

        if not preserved_file.exists():
            shutil.copy2(
                source_file,
                preserved_file
            )

        shutil.copy2(
            source_file,
            output_file
        )

    else:
        raise RuntimeError(
            f"No acquisition handler implemented for {dataset_id}"
        )

    # --------------------------------------------------------------------------
    # Final verification
    # --------------------------------------------------------------------------

    if not output_file.exists():
        raise RuntimeError(
            f"Expected final raw dataset was not created:\n{output_file}"
        )

    if output_file.stat().st_size == 0:
        raise RuntimeError(
            f"Final raw dataset is empty:\n{output_file}"
        )

    record["raw_file_sha256"] = calculate_sha256(output_file)
    record["status"] = "SUCCESS"

    ACQUISITION_MANIFEST["datasets"][dataset_id] = record

    print()
    print("✓ Final raw dataset:")
    print(f"  {output_file}")
    print(f"✓ SHA-256:")
    print(f"  {record['raw_file_sha256']}")

# ------------------------------------------------------------------------------
# Save acquisition manifest
# ------------------------------------------------------------------------------

acquisition_manifest_path = (
    MANIFEST_DIR / "raw_dataset_download_manifest.json"
)

with open(
    acquisition_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        ACQUISITION_MANIFEST,
        f,
        indent=2
    )

print()
print("=" * 100)
print("01.5 ACQUISITION COMPLETE")
print("=" * 100)

for dataset_id, record in ACQUISITION_MANIFEST["datasets"].items():
    print(
        f"{dataset_id:20s} : "
        f"{record['status']:20s} | "
        f"{record['raw_file_sha256']}"
    )

print()
print(f"✓ Acquisition manifest saved:")
print(f"  {acquisition_manifest_path}")

01.5 — OFFICIAL RAW DATASET ACQUISITION & PRESERVATION
✓ Raw data directory : /content/drive/MyDrive/SPP_GAN_Research/data/raw
✓ Download directory : /content/drive/MyDrive/SPP_GAN_Research/tmp/raw_dataset_downloads
✓ Manifest directory : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/manifest

----------------------------------------------------------------------------------------------------
DATASET: adult_income
----------------------------------------------------------------------------------------------------
✓ Final raw dataset already exists:
  /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv

----------------------------------------------------------------------------------------------------
DATASET: bank_marketing
----------------------------------------------------------------------------------------------------
✓ Final raw dataset already exists:
  /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv

-------------------

In [21]:
# ==============================================================================
# 01.6 — RAM-SAFE DATASET LOADER
# ==============================================================================

print("=" * 100)
print("01.6 — RAM-SAFE DATASET LOADER")
print("=" * 100)

def load_raw_dataset(dataset_id):
    """
    Load exactly one raw dataset into memory.

    No transformations are performed.
    """

    if dataset_id not in RAW_OUTPUT_FILES:
        raise KeyError(f"Unknown dataset_id: {dataset_id}")

    file_path = RAW_OUTPUT_FILES[dataset_id]

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    print(f"Loading dataset: {dataset_id}")
    print(f"File          : {file_path}")

    start_time = time.time()

    # --------------------------------------------------------------------------
    # Dataset-specific source format
    # --------------------------------------------------------------------------

    if dataset_id == "adult_income":
        df = pd.read_csv(
            file_path,
            low_memory=False
        )

    elif dataset_id == "bank_marketing":
        df = pd.read_csv(
            file_path,
            sep=";",
            low_memory=False
        )

    elif dataset_id == "diabetes_130us":
        df = pd.read_csv(
            file_path,
            low_memory=False
        )

    else:
        raise RuntimeError(
            f"No raw loader configured for {dataset_id}"
        )

    elapsed = time.time() - start_time

    print(
        f"✓ Loaded {len(df):,} rows × {df.shape[1]:,} columns "
        f"in {elapsed:.2f} seconds"
    )

    return df

print("✓ RAM-safe loader initialized.")

01.6 — RAM-SAFE DATASET LOADER
✓ RAM-safe loader initialized.


In [22]:
# ==============================================================================
# 01.7 — RAW DATASET STRUCTURAL VALIDATION
# ==============================================================================

print("=" * 100)
print("01.7 — RAW DATASET STRUCTURAL VALIDATION")
print("=" * 100)

STRUCTURAL_REPORTS = {}
DATASET_SHAPES = {}

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"STRUCTURAL VALIDATION: {dataset_id}")
    print("-" * 100)

    df = load_raw_dataset(dataset_id)

    rows, columns = df.shape

    DATASET_SHAPES[dataset_id] = {
        "rows": int(rows),
        "columns": int(columns),
    }

    report = {
        "dataset_id": dataset_id,
        "rows": int(rows),
        "columns": int(columns),
        "column_names": [str(c) for c in df.columns],
        "duplicate_column_names": bool(
            df.columns.duplicated().any()
        ),
        "empty_column_names": [
            str(c)
            for c in df.columns
            if str(c).strip() == ""
        ],
        "all_null_columns": [
            str(c)
            for c in df.columns
            if df[c].isna().all()
        ],
        "all_null_rows": int(df.isna().all(axis=1).sum()),
        "memory_usage_mb": float(
            df.memory_usage(deep=True).sum() / (1024 ** 2)
        ),
        "status": "PASS",
    }

    if rows == 0:
        report["status"] = "FAIL"

    if columns == 0:
        report["status"] = "FAIL"

    if report["duplicate_column_names"]:
        report["status"] = "FAIL"

    STRUCTURAL_REPORTS[dataset_id] = report

    print(f"Rows       : {rows:,}")
    print(f"Columns    : {columns:,}")
    print(f"Memory     : {report['memory_usage_mb']:.2f} MB")
    print(f"Duplicates : {report['duplicate_column_names']}")
    print(f"Status     : {report['status']}")

    del df

print()
print("✓ Structural validation completed.")

01.7 — RAW DATASET STRUCTURAL VALIDATION

----------------------------------------------------------------------------------------------------
STRUCTURAL VALIDATION: adult_income
----------------------------------------------------------------------------------------------------
Loading dataset: adult_income
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
✓ Loaded 48,842 rows × 15 columns in 0.45 seconds
Rows       : 48,842
Columns    : 15
Memory     : 26.36 MB
Duplicates : False
Status     : PASS

----------------------------------------------------------------------------------------------------
STRUCTURAL VALIDATION: bank_marketing
----------------------------------------------------------------------------------------------------
Loading dataset: bank_marketing
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
✓ Loaded 45,211 rows × 17 columns in 0.09 seconds
Rows       : 45,211
Columns    : 17
Memory     : 25.75 M

In [23]:
# ==============================================================================
# 01.8 — TARGET COLUMN RESOLUTION
# ==============================================================================

print("=" * 100)
print("01.8 — TARGET COLUMN RESOLUTION")
print("=" * 100)

TARGET_COLUMN_CANDIDATES = {
    "adult_income": [
        "income",
    ],
    "bank_marketing": [
        "y",
    ],
    "diabetes_130us": [
        "readmitted",
    ],
}

TARGET_COLUMNS = {}

for dataset_id in DATASET_IDS:

    df = load_raw_dataset(dataset_id)

    candidates = TARGET_COLUMN_CANDIDATES.get(
        dataset_id,
        []
    )

    normalized_columns = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    matched = []

    for candidate in candidates:
        normalized_candidate = candidate.strip().lower()

        if normalized_candidate in normalized_columns:
            matched.append(
                normalized_columns[normalized_candidate]
            )

    if len(matched) != 1:
        raise RuntimeError(
            f"Target-column resolution failed for {dataset_id}.\n"
            f"Candidates: {candidates}\n"
            f"Matched: {matched}\n"
            f"Available columns: {list(df.columns)}"
        )

    TARGET_COLUMNS[dataset_id] = matched[0]

    print(
        f"{dataset_id:20s} → target = "
        f"{TARGET_COLUMNS[dataset_id]}"
    )

    del df

print()
print("✓ Target columns resolved.")

01.8 — TARGET COLUMN RESOLUTION
Loading dataset: adult_income
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
✓ Loaded 48,842 rows × 15 columns in 0.12 seconds
adult_income         → target = income
Loading dataset: bank_marketing
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
✓ Loaded 45,211 rows × 17 columns in 0.10 seconds
bank_marketing       → target = y
Loading dataset: diabetes_130us
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/diabetes_130us.csv
✓ Loaded 101,766 rows × 50 columns in 1.07 seconds
diabetes_130us       → target = readmitted

✓ Target columns resolved.


In [24]:
# ==============================================================================
# 01.9 — RAW MISSING-VALUE ANALYSIS
# ==============================================================================

print("=" * 100)
print("01.9 — RAW MISSING-VALUE ANALYSIS")
print("=" * 100)

MISSINGNESS_REPORTS = {}

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"MISSINGNESS: {dataset_id}")
    print("-" * 100)

    df = load_raw_dataset(dataset_id)

    records = []

    n_rows = len(df)

    for column in df.columns:

        missing_count = int(df[column].isna().sum())

        # Also identify common raw-data missing tokens without changing data.
        string_series = df[column].astype("string")

        stripped = string_series.str.strip()

        raw_missing_tokens = {
            "?": int((stripped == "?").sum()),
            "": int((stripped == "").sum()),
            "NA": int(
                stripped.str.upper().eq("NA").sum()
            ),
            "N/A": int(
                stripped.str.upper().eq("N/A").sum()
            ),
            "NULL": int(
                stripped.str.upper().eq("NULL").sum()
            ),
        }

        records.append({
            "dataset_id": dataset_id,
            "feature": str(column),
            "dtype": str(df[column].dtype),
            "missing_count": missing_count,
            "missing_rate": (
                missing_count / n_rows
                if n_rows > 0
                else np.nan
            ),
            "question_mark_count": raw_missing_tokens["?"],
            "empty_string_count": raw_missing_tokens[""],
            "na_token_count": raw_missing_tokens["NA"],
            "n_a_token_count": raw_missing_tokens["N/A"],
            "null_token_count": raw_missing_tokens["NULL"],
        })

    report_df = pd.DataFrame(records)

    MISSINGNESS_REPORTS[dataset_id] = report_df

    print(
        f"Features with pandas NA: "
        f"{(report_df['missing_count'] > 0).sum()}"
    )

    print(
        f"Features containing '?' token: "
        f"{(report_df['question_mark_count'] > 0).sum()}"
    )

    print(
        f"Total pandas NA values: "
        f"{report_df['missing_count'].sum():,}"
    )

    del df

print()
print("✓ Raw missing-value analysis completed.")

01.9 — RAW MISSING-VALUE ANALYSIS

----------------------------------------------------------------------------------------------------
MISSINGNESS: adult_income
----------------------------------------------------------------------------------------------------
Loading dataset: adult_income
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
✓ Loaded 48,842 rows × 15 columns in 0.13 seconds
Features with pandas NA: 3
Features containing '?' token: 0
Total pandas NA values: 6,465

----------------------------------------------------------------------------------------------------
MISSINGNESS: bank_marketing
----------------------------------------------------------------------------------------------------
Loading dataset: bank_marketing
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
✓ Loaded 45,211 rows × 17 columns in 0.06 seconds
Features with pandas NA: 0
Features containing '?' token: 0
Total pandas NA values: 0

-

In [25]:
# ==============================================================================
# 01.10 — DUPLICATE RECORD ANALYSIS
# ==============================================================================

print("=" * 100)
print("01.10 — DUPLICATE RECORD ANALYSIS")
print("=" * 100)

DUPLICATE_REPORTS = {}

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"DUPLICATE ANALYSIS: {dataset_id}")
    print("-" * 100)

    df = load_raw_dataset(dataset_id)

    duplicate_mask = df.duplicated(
        keep=False
    )

    duplicate_row_count = int(
        duplicate_mask.sum()
    )

    duplicate_group_count = int(
        df.duplicated(
            keep="first"
        ).sum()
    )

    report = {
        "dataset_id": dataset_id,
        "rows": int(len(df)),
        "duplicate_rows_including_first": duplicate_row_count,
        "duplicate_rows_excluding_first": duplicate_group_count,
        "duplicate_rate": (
            duplicate_group_count / len(df)
            if len(df) > 0
            else np.nan
        ),
        "status": "PASS",
    }

    DUPLICATE_REPORTS[dataset_id] = report

    print(
        f"Rows                              : {len(df):,}"
    )
    print(
        f"Duplicate rows including first    : "
        f"{duplicate_row_count:,}"
    )
    print(
        f"Duplicate rows excluding first    : "
        f"{duplicate_group_count:,}"
    )
    print(
        f"Duplicate rate                    : "
        f"{report['duplicate_rate']:.6f}"
    )

    del df

print()
print("✓ Duplicate analysis completed.")
print("  NOTE: No duplicates were removed.")

01.10 — DUPLICATE RECORD ANALYSIS

----------------------------------------------------------------------------------------------------
DUPLICATE ANALYSIS: adult_income
----------------------------------------------------------------------------------------------------
Loading dataset: adult_income
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
✓ Loaded 48,842 rows × 15 columns in 0.08 seconds
Rows                              : 48,842
Duplicate rows including first    : 101
Duplicate rows excluding first    : 52
Duplicate rate                    : 0.001065

----------------------------------------------------------------------------------------------------
DUPLICATE ANALYSIS: bank_marketing
----------------------------------------------------------------------------------------------------
Loading dataset: bank_marketing
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
✓ Loaded 45,211 rows × 17 columns in 0.08 secon

In [26]:
# ==============================================================================
# 01.11 — CONSTANT / ZERO-VARIANCE FEATURE DETECTION
# ==============================================================================

print("=" * 100)
print("01.11 — CONSTANT / ZERO-VARIANCE FEATURE DETECTION")
print("=" * 100)

CONSTANT_FEATURE_REPORTS = {}

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"CONSTANT FEATURES: {dataset_id}")
    print("-" * 100)

    df = load_raw_dataset(dataset_id)

    records = []

    for column in df.columns:

        series = df[column]

        unique_non_null = int(
            series.nunique(
                dropna=True
            )
        )

        is_constant = unique_non_null <= 1

        variance = np.nan

        if pd.api.types.is_numeric_dtype(series):

            if unique_non_null > 1:
                variance = float(
                    series.var(
                        skipna=True
                    )
                )
            elif unique_non_null == 1:
                variance = 0.0

        records.append({
            "dataset_id": dataset_id,
            "feature": str(column),
            "dtype": str(series.dtype),
            "unique_non_null": unique_non_null,
            "variance": variance,
            "is_constant": bool(is_constant),
        })

    report_df = pd.DataFrame(records)

    CONSTANT_FEATURE_REPORTS[dataset_id] = report_df

    print(
        f"Constant / single-value features: "
        f"{report_df['is_constant'].sum()}"
    )

    for feature in report_df.loc[
        report_df["is_constant"],
        "feature"
    ]:
        print(f"  - {feature}")

    del df

print()
print("✓ Constant-feature detection completed.")
print("  NOTE: No features were removed.")

01.11 — CONSTANT / ZERO-VARIANCE FEATURE DETECTION

----------------------------------------------------------------------------------------------------
CONSTANT FEATURES: adult_income
----------------------------------------------------------------------------------------------------
Loading dataset: adult_income
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
✓ Loaded 48,842 rows × 15 columns in 0.54 seconds
Constant / single-value features: 0

----------------------------------------------------------------------------------------------------
CONSTANT FEATURES: bank_marketing
----------------------------------------------------------------------------------------------------
Loading dataset: bank_marketing
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
✓ Loaded 45,211 rows × 17 columns in 0.31 seconds
Constant / single-value features: 0

----------------------------------------------------------------------------

In [27]:
# ==============================================================================
# 01.12 — IDENTIFIER CANDIDATE DETECTION
# ==============================================================================

print("=" * 100)
print("01.12 — IDENTIFIER CANDIDATE DETECTION")
print("=" * 100)

IDENTIFIER_REPORTS = {}

IDENTIFIER_NAME_PATTERNS = [
    r"^id$",
    r"_id$",
    r"^id_",
    r"identifier",
    r"patient.?number",
    r"patient.?id",
    r"customer.?id",
    r"account.?id",
    r"record.?id",
    r"row.?id",
]

def name_looks_like_identifier(column_name):

    normalized = (
        str(column_name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    return any(
        re.search(pattern, normalized)
        for pattern in IDENTIFIER_NAME_PATTERNS
    )

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"IDENTIFIER ANALYSIS: {dataset_id}")
    print("-" * 100)

    df = load_raw_dataset(dataset_id)

    records = []

    n_rows = len(df)

    for column in df.columns:

        series = df[column]

        unique_count = int(
            series.nunique(
                dropna=True
            )
        )

        unique_ratio = (
            unique_count / n_rows
            if n_rows > 0
            else np.nan
        )

        name_signal = name_looks_like_identifier(
            column
        )

        near_unique_signal = (
            unique_ratio >= 0.98
            if pd.notna(unique_ratio)
            else False
        )

        identifier_candidate = (
            name_signal or near_unique_signal
        )

        records.append({
            "dataset_id": dataset_id,
            "feature": str(column),
            "dtype": str(series.dtype),
            "unique_count": unique_count,
            "unique_ratio": unique_ratio,
            "name_identifier_signal": bool(name_signal),
            "near_unique_signal": bool(
                near_unique_signal
            ),
            "identifier_candidate": bool(
                identifier_candidate
            ),
        })

    report_df = pd.DataFrame(records)

    IDENTIFIER_REPORTS[dataset_id] = report_df

    candidates = report_df[
        report_df["identifier_candidate"]
    ]

    print(
        f"Identifier candidates: {len(candidates)}"
    )

    if len(candidates) > 0:

        for _, row in candidates.iterrows():

            print(
                f"  - {row['feature']} "
                f"(unique ratio={row['unique_ratio']:.6f})"
            )

    del df

print()
print("✓ Identifier-candidate detection completed.")
print("  NOTE: Candidate identifiers were NOT removed.")

01.12 — IDENTIFIER CANDIDATE DETECTION

----------------------------------------------------------------------------------------------------
IDENTIFIER ANALYSIS: adult_income
----------------------------------------------------------------------------------------------------
Loading dataset: adult_income
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
✓ Loaded 48,842 rows × 15 columns in 0.09 seconds
Identifier candidates: 0

----------------------------------------------------------------------------------------------------
IDENTIFIER ANALYSIS: bank_marketing
----------------------------------------------------------------------------------------------------
Loading dataset: bank_marketing
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
✓ Loaded 45,211 rows × 17 columns in 0.08 seconds
Identifier candidates: 0

----------------------------------------------------------------------------------------------------
IDENT

In [28]:
# ==============================================================================
# 01.13 — NUMERIC / CATEGORICAL FEATURE INVENTORY
# ==============================================================================

print("=" * 100)
print("01.13 — NUMERIC / CATEGORICAL FEATURE INVENTORY")
print("=" * 100)

FEATURE_INVENTORIES = {}

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"FEATURE INVENTORY: {dataset_id}")
    print("-" * 100)

    df = load_raw_dataset(dataset_id)

    records = []

    target_column = TARGET_COLUMNS[dataset_id]

    for column in df.columns:

        series = df[column]

        is_target = (
            str(column) == str(target_column)
        )

        is_numeric = (
            pd.api.types.is_numeric_dtype(series)
        )

        if is_numeric:
            feature_type = "numeric"
        else:
            feature_type = "categorical"

        records.append({
            "dataset_id": dataset_id,
            "feature": str(column),
            "dtype": str(series.dtype),
            "feature_type": feature_type,
            "is_target": bool(is_target),
            "n_unique": int(
                series.nunique(
                    dropna=True
                )
            ),
            "n_missing": int(
                series.isna().sum()
            ),
            "missing_rate": (
                float(series.isna().mean())
                if len(series) > 0
                else np.nan
            ),
        })

    inventory_df = pd.DataFrame(records)

    FEATURE_INVENTORIES[dataset_id] = inventory_df

    print(
        f"Numeric features    : "
        f"{(inventory_df['feature_type'] == 'numeric').sum()}"
    )

    print(
        f"Categorical features : "
        f"{(inventory_df['feature_type'] == 'categorical').sum()}"
    )

    print(
        f"Target               : "
        f"{target_column}"
    )

    del df

print()
print("✓ Feature inventories completed.")

01.13 — NUMERIC / CATEGORICAL FEATURE INVENTORY

----------------------------------------------------------------------------------------------------
FEATURE INVENTORY: adult_income
----------------------------------------------------------------------------------------------------
Loading dataset: adult_income
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
✓ Loaded 48,842 rows × 15 columns in 0.45 seconds
Numeric features    : 6
Categorical features : 9
Target               : income

----------------------------------------------------------------------------------------------------
FEATURE INVENTORY: bank_marketing
----------------------------------------------------------------------------------------------------
Loading dataset: bank_marketing
File          : /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
✓ Loaded 45,211 rows × 17 columns in 0.30 seconds
Numeric features    : 7
Categorical features : 10
Target               : 

In [29]:
# ==============================================================================
# 01.14 — RAW FILE SHA-256 FINGERPRINTING
# ==============================================================================

print("=" * 100)
print("01.14 — RAW FILE SHA-256 FINGERPRINTING")
print("=" * 100)

RAW_FILE_FINGERPRINTS = {}

for dataset_id in DATASET_IDS:

    file_path = RAW_OUTPUT_FILES[dataset_id]

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    sha256 = calculate_sha256(file_path)

    fingerprint = {
        "dataset_id": dataset_id,
        "file_name": file_path.name,
        "absolute_path": str(file_path),
        "file_size_bytes": int(
            file_path.stat().st_size
        ),
        "sha256": sha256,
        "fingerprint_timestamp_utc": (
            datetime.now(timezone.utc).isoformat()
        ),
    }

    RAW_FILE_FINGERPRINTS[dataset_id] = fingerprint

    print()
    print(dataset_id)
    print(f"  File size : {fingerprint['file_size_bytes']:,} bytes")
    print(f"  SHA-256   : {sha256}")

print()
print("✓ Raw file fingerprints generated.")

01.14 — RAW FILE SHA-256 FINGERPRINTING

adult_income
  File size : 5,271,057 bytes
  SHA-256   : 9479f8b76861e48c66836d97937f2c5a469581672e12c795d770b297817ae3a1

bank_marketing
  File size : 4,610,348 bytes
  SHA-256   : d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d

diabetes_130us
  File size : 19,159,383 bytes
  SHA-256   : 0689e7ec031237dc63031b938805c48377748761a3b26acab621567afa24df97

✓ Raw file fingerprints generated.


In [30]:
# ==============================================================================
# 01.15 — FULL RAW DATASET VALIDATION ENGINE
# ==============================================================================

print("=" * 100)
print("01.15 — FULL RAW DATASET VALIDATION ENGINE")
print("=" * 100)

DATASET_VALIDATION_RESULTS = {}

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"FULL VALIDATION: {dataset_id}")
    print("-" * 100)

    checks = {}

    file_path = RAW_OUTPUT_FILES[dataset_id]

    # --------------------------------------------------------------------------
    # File checks
    # --------------------------------------------------------------------------

    checks["raw_file_exists"] = file_path.exists()

    checks["raw_file_nonempty"] = (
        file_path.exists()
        and file_path.stat().st_size > 0
    )

    # --------------------------------------------------------------------------
    # Structural checks
    # --------------------------------------------------------------------------

    structural = STRUCTURAL_REPORTS[dataset_id]

    checks["rows_positive"] = (
        structural["rows"] > 0
    )

    checks["columns_positive"] = (
        structural["columns"] > 0
    )

    checks["unique_column_names"] = (
        not structural["duplicate_column_names"]
    )

    checks["target_resolved"] = (
        dataset_id in TARGET_COLUMNS
        and TARGET_COLUMNS[dataset_id]
        in structural["column_names"]
    )

    checks["no_all_null_columns"] = (
        len(structural["all_null_columns"]) == 0
    )

    # --------------------------------------------------------------------------
    # Fingerprint
    # --------------------------------------------------------------------------

    checks["sha256_available"] = (
        dataset_id in RAW_FILE_FINGERPRINTS
        and len(
            RAW_FILE_FINGERPRINTS[dataset_id]["sha256"]
        ) == 64
    )

    # --------------------------------------------------------------------------
    # Determine final status
    # --------------------------------------------------------------------------

    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    status = (
        "PASS"
        if len(failed_checks) == 0
        else "FAIL"
    )

    DATASET_VALIDATION_RESULTS[dataset_id] = {
        "dataset_id": dataset_id,
        "status": status,
        "checks": checks,
        "failed_checks": failed_checks,
        "rows": structural["rows"],
        "columns": structural["columns"],
        "target_column": TARGET_COLUMNS[dataset_id],
        "sha256": RAW_FILE_FINGERPRINTS[dataset_id]["sha256"],
    }

    print(f"Status : {status}")

    if failed_checks:
        print("Failed checks:")
        for check in failed_checks:
            print(f"  ✗ {check}")
    else:
        print("All validation checks passed.")

print()
print("✓ Full raw-data validation completed.")

01.15 — FULL RAW DATASET VALIDATION ENGINE

----------------------------------------------------------------------------------------------------
FULL VALIDATION: adult_income
----------------------------------------------------------------------------------------------------
Status : PASS
All validation checks passed.

----------------------------------------------------------------------------------------------------
FULL VALIDATION: bank_marketing
----------------------------------------------------------------------------------------------------
Status : PASS
All validation checks passed.

----------------------------------------------------------------------------------------------------
FULL VALIDATION: diabetes_130us
----------------------------------------------------------------------------------------------------
Status : PASS
All validation checks passed.

✓ Full raw-data validation completed.


In [31]:
# ==============================================================================
# 01.16 — DATASET VALIDATION SUMMARY
# ==============================================================================

print("=" * 100)
print("01.16 — DATASET VALIDATION SUMMARY")
print("=" * 100)

DATASET_VALIDATION_SUMMARY = []

for dataset_id in DATASET_IDS:

    result = DATASET_VALIDATION_RESULTS[dataset_id]

    DATASET_VALIDATION_SUMMARY.append({
        "dataset_id": dataset_id,
        "rows": result["rows"],
        "columns": result["columns"],
        "target_column": result["target_column"],
        "sha256": result["sha256"],
        "status": result["status"],
        "failed_check_count": len(
            result["failed_checks"]
        ),
    })

DATASET_VALIDATION_SUMMARY_DF = pd.DataFrame(
    DATASET_VALIDATION_SUMMARY
)

display(DATASET_VALIDATION_SUMMARY_DF)

if not (
    DATASET_VALIDATION_SUMMARY_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more datasets failed raw-data validation."
    )

print()
print("✓ All datasets passed validation.")

01.16 — DATASET VALIDATION SUMMARY


,dataset_id,rows,columns,target_column,sha256,status,failed_check_count
0,adult_income,48842,15,income,9479f8b76861e48c66836d97937f2c5a469581672e12c7...,PASS,0
1,bank_marketing,45211,17,y,d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa...,PASS,0
2,diabetes_130us,101766,50,readmitted,0689e7ec031237dc63031b938805c48377748761a3b26a...,PASS,0



✓ All datasets passed validation.


In [32]:
# ==============================================================================
# 01.17 — RAW MISSINGNESS SUMMARY
# ==============================================================================

print("=" * 100)
print("01.17 — RAW MISSINGNESS SUMMARY")
print("=" * 100)

RAW_MISSINGNESS_SUMMARY = []

for dataset_id in DATASET_IDS:

    report_df = MISSINGNESS_REPORTS[dataset_id]

    RAW_MISSINGNESS_SUMMARY.append({
        "dataset_id": dataset_id,
        "features": len(report_df),
        "features_with_pandas_na": int(
            (
                report_df["missing_count"] > 0
            ).sum()
        ),
        "total_pandas_na": int(
            report_df["missing_count"].sum()
        ),
        "features_with_question_mark": int(
            (
                report_df["question_mark_count"] > 0
            ).sum()
        ),
        "total_question_mark_tokens": int(
            report_df["question_mark_count"].sum()
        ),
        "max_missing_rate": float(
            report_df["missing_rate"].max()
        ),
    })

RAW_MISSINGNESS_SUMMARY_DF = pd.DataFrame(
    RAW_MISSINGNESS_SUMMARY
)

display(RAW_MISSINGNESS_SUMMARY_DF)

print("✓ Missingness summary generated.")

01.17 — RAW MISSINGNESS SUMMARY


,dataset_id,features,features_with_pandas_na,total_pandas_na,features_with_question_mark,total_question_mark_tokens,max_missing_rate
0,adult_income,15,3,6465,0,0,0.057512
1,bank_marketing,17,0,0,0,0,0.000000
2,diabetes_130us,50,2,181168,7,192849,0.947468


✓ Missingness summary generated.


In [33]:
# ==============================================================================
# 01.18 — DUPLICATE / CONSTANT / IDENTIFIER SUMMARY
# ==============================================================================

print("=" * 100)
print("01.18 — DUPLICATE / CONSTANT / IDENTIFIER SUMMARY")
print("=" * 100)

STRUCTURAL_ANOMALY_SUMMARY = []

for dataset_id in DATASET_IDS:

    duplicate = DUPLICATE_REPORTS[dataset_id]

    constant_df = CONSTANT_FEATURE_REPORTS[dataset_id]

    identifier_df = IDENTIFIER_REPORTS[dataset_id]

    STRUCTURAL_ANOMALY_SUMMARY.append({
        "dataset_id": dataset_id,

        "duplicate_rows_excluding_first": (
            duplicate[
                "duplicate_rows_excluding_first"
            ]
        ),

        "duplicate_rate": duplicate[
            "duplicate_rate"
        ],

        "constant_feature_count": int(
            constant_df["is_constant"].sum()
        ),

        "identifier_candidate_count": int(
            identifier_df[
                "identifier_candidate"
            ].sum()
        ),
    })

STRUCTURAL_ANOMALY_SUMMARY_DF = pd.DataFrame(
    STRUCTURAL_ANOMALY_SUMMARY
)

display(STRUCTURAL_ANOMALY_SUMMARY_DF)

print("✓ Structural anomaly summary generated.")
print()
print("IMPORTANT:")
print("  These characteristics are reported only.")
print("  No rows or features have been removed.")

01.18 — DUPLICATE / CONSTANT / IDENTIFIER SUMMARY


,dataset_id,duplicate_rows_excluding_first,duplicate_rate,constant_feature_count,identifier_candidate_count
0,adult_income,52,0.001065,0,0
1,bank_marketing,0,0.000000,0,0
2,diabetes_130us,0,0.000000,2,4


✓ Structural anomaly summary generated.

IMPORTANT:
  These characteristics are reported only.
  No rows or features have been removed.


In [34]:
# ==============================================================================
# 01.19 — SAVE FEATURE INVENTORIES
# ==============================================================================

print("=" * 100)
print("01.19 — SAVE FEATURE INVENTORIES")
print("=" * 100)

FEATURE_INVENTORY_DIR = NOTEBOOK_01_DIR / "feature_inventory"
FEATURE_INVENTORY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_INVENTORY_PATHS = {}

for dataset_id, inventory_df in FEATURE_INVENTORIES.items():

    output_path = (
        FEATURE_INVENTORY_DIR
        / f"{dataset_id}_feature_inventory.csv"
    )

    inventory_df.to_csv(
        output_path,
        index=False
    )

    FEATURE_INVENTORY_PATHS[dataset_id] = str(
        output_path
    )

    print(
        f"✓ {dataset_id}: {output_path}"
    )

print()
print("✓ Feature inventories saved.")

01.19 — SAVE FEATURE INVENTORIES
✓ adult_income: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/feature_inventory/adult_income_feature_inventory.csv
✓ bank_marketing: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/feature_inventory/bank_marketing_feature_inventory.csv
✓ diabetes_130us: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/feature_inventory/diabetes_130us_feature_inventory.csv

✓ Feature inventories saved.


In [35]:
# ==============================================================================
# 01.20 — SAVE MISSINGNESS REPORTS
# ==============================================================================

print("=" * 100)
print("01.20 — SAVE MISSINGNESS REPORTS")
print("=" * 100)

MISSINGNESS_DIR = NOTEBOOK_01_DIR / "missingness"
MISSINGNESS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MISSINGNESS_PATHS = {}

for dataset_id, report_df in MISSINGNESS_REPORTS.items():

    output_path = (
        MISSINGNESS_DIR
        / f"{dataset_id}_raw_missingness.csv"
    )

    report_df.to_csv(
        output_path,
        index=False
    )

    MISSINGNESS_PATHS[dataset_id] = str(
        output_path
    )

    print(
        f"✓ {dataset_id}: {output_path}"
    )

print()
print("✓ Missingness reports saved.")

01.20 — SAVE MISSINGNESS REPORTS
✓ adult_income: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/missingness/adult_income_raw_missingness.csv
✓ bank_marketing: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/missingness/bank_marketing_raw_missingness.csv
✓ diabetes_130us: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/missingness/diabetes_130us_raw_missingness.csv

✓ Missingness reports saved.


In [36]:
# ==============================================================================
# 01.21 — SAVE STRUCTURAL REPORTS
# ==============================================================================

print("=" * 100)
print("01.21 — SAVE STRUCTURAL REPORTS")
print("=" * 100)

STRUCTURAL_DIR = NOTEBOOK_01_DIR / "structural"
STRUCTURAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DUPLICATE_PATHS = {}
CONSTANT_PATHS = {}
IDENTIFIER_PATHS = {}

for dataset_id in DATASET_IDS:

    # --------------------------------------------------------------------------
    # Duplicate report
    # --------------------------------------------------------------------------

    duplicate_path = (
        STRUCTURAL_DIR
        / f"{dataset_id}_duplicate_summary.json"
    )

    with open(
        duplicate_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            DUPLICATE_REPORTS[dataset_id],
            f,
            indent=2
        )

    DUPLICATE_PATHS[dataset_id] = str(
        duplicate_path
    )

    # --------------------------------------------------------------------------
    # Constant report
    # --------------------------------------------------------------------------

    constant_path = (
        STRUCTURAL_DIR
        / f"{dataset_id}_constant_features.csv"
    )

    CONSTANT_FEATURE_REPORTS[
        dataset_id
    ].to_csv(
        constant_path,
        index=False
    )

    CONSTANT_PATHS[dataset_id] = str(
        constant_path
    )

    # --------------------------------------------------------------------------
    # Identifier report
    # --------------------------------------------------------------------------

    identifier_path = (
        STRUCTURAL_DIR
        / f"{dataset_id}_identifier_candidates.csv"
    )

    IDENTIFIER_REPORTS[
        dataset_id
    ].to_csv(
        identifier_path,
        index=False
    )

    IDENTIFIER_PATHS[dataset_id] = str(
        identifier_path
    )

print("✓ Duplicate reports saved.")
print("✓ Constant-feature reports saved.")
print("✓ Identifier-candidate reports saved.")

01.21 — SAVE STRUCTURAL REPORTS
✓ Duplicate reports saved.
✓ Constant-feature reports saved.
✓ Identifier-candidate reports saved.


In [37]:
# ==============================================================================
# 01.22 — SAVE DATASET-LEVEL VALIDATION SUMMARY
# ==============================================================================

print("=" * 100)
print("01.22 — SAVE DATASET-LEVEL VALIDATION SUMMARY")
print("=" * 100)

VALIDATION_DIR = NOTEBOOK_01_DIR / "validation"
VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_VALIDATION_SUMMARY_PATH = (
    VALIDATION_DIR
    / "dataset_validation_summary.csv"
)

DATASET_VALIDATION_SUMMARY_DF.to_csv(
    DATASET_VALIDATION_SUMMARY_PATH,
    index=False
)

RAW_MISSINGNESS_SUMMARY_PATH = (
    VALIDATION_DIR
    / "raw_missingness_summary.csv"
)

RAW_MISSINGNESS_SUMMARY_DF.to_csv(
    RAW_MISSINGNESS_SUMMARY_PATH,
    index=False
)

STRUCTURAL_ANOMALY_SUMMARY_PATH = (
    VALIDATION_DIR
    / "structural_anomaly_summary.csv"
)

STRUCTURAL_ANOMALY_SUMMARY_DF.to_csv(
    STRUCTURAL_ANOMALY_SUMMARY_PATH,
    index=False
)

print(
    f"✓ Dataset validation summary:\n"
    f"  {DATASET_VALIDATION_SUMMARY_PATH}"
)

print(
    f"✓ Missingness summary:\n"
    f"  {RAW_MISSINGNESS_SUMMARY_PATH}"
)

print(
    f"✓ Structural summary:\n"
    f"  {STRUCTURAL_ANOMALY_SUMMARY_PATH}"
)

01.22 — SAVE DATASET-LEVEL VALIDATION SUMMARY
✓ Dataset validation summary:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/validation/dataset_validation_summary.csv
✓ Missingness summary:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/validation/raw_missingness_summary.csv
✓ Structural summary:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/validation/structural_anomaly_summary.csv


In [38]:
# ==============================================================================
# 01.23 — BUILD VALIDATED DATASET REGISTRY
# ==============================================================================

print("=" * 100)
print("01.23 — BUILD VALIDATED DATASET REGISTRY")
print("=" * 100)

VALIDATED_DATASET_REGISTRY = {}

for dataset_id in DATASET_IDS:

    validation = DATASET_VALIDATION_RESULTS[
        dataset_id
    ]

    fingerprint = RAW_FILE_FINGERPRINTS[
        dataset_id
    ]

    feature_inventory = FEATURE_INVENTORIES[
        dataset_id
    ]

    validated_record = {
        "dataset_id": dataset_id,

        "raw_file": str(
            RAW_OUTPUT_FILES[dataset_id]
        ),

        "raw_file_name": (
            RAW_OUTPUT_FILES[dataset_id].name
        ),

        "raw_file_sha256": fingerprint[
            "sha256"
        ],

        "raw_file_size_bytes": fingerprint[
            "file_size_bytes"
        ],

        "rows": int(
            validation["rows"]
        ),

        "columns": int(
            validation["columns"]
        ),

        "target_column": str(
            validation["target_column"]
        ),

        "numeric_features": [
            str(x)
            for x in feature_inventory.loc[
                feature_inventory["feature_type"]
                == "numeric",
                "feature"
            ].tolist()
        ],

        "categorical_features": [
            str(x)
            for x in feature_inventory.loc[
                feature_inventory["feature_type"]
                == "categorical",
                "feature"
            ].tolist()
        ],

        "feature_count_excluding_target": int(
            len(feature_inventory)
            - feature_inventory[
                "is_target"
            ].sum()
        ),

        "validation_status": validation[
            "status"
        ],

        "source": "UCI Machine Learning Repository",

        "notebook_01_validation_timestamp_utc": (
            datetime.now(timezone.utc).isoformat()
        ),
    }

    VALIDATED_DATASET_REGISTRY[
        dataset_id
    ] = validated_record

print("✓ Validated dataset registry constructed.")

for dataset_id, record in VALIDATED_DATASET_REGISTRY.items():

    print()
    print(dataset_id)
    print(
        f"  Rows        : {record['rows']:,}"
    )
    print(
        f"  Columns     : {record['columns']:,}"
    )
    print(
        f"  Target      : {record['target_column']}"
    )
    print(
        f"  Num features: "
        f"{len(record['numeric_features'])}"
    )
    print(
        f"  Cat features: "
        f"{len(record['categorical_features'])}"
    )
    print(
        f"  Validation  : "
        f"{record['validation_status']}"
    )

01.23 — BUILD VALIDATED DATASET REGISTRY
✓ Validated dataset registry constructed.

adult_income
  Rows        : 48,842
  Columns     : 15
  Target      : income
  Num features: 6
  Cat features: 9
  Validation  : PASS

bank_marketing
  Rows        : 45,211
  Columns     : 17
  Target      : y
  Num features: 7
  Cat features: 10
  Validation  : PASS

diabetes_130us
  Rows        : 101,766
  Columns     : 50
  Target      : readmitted
  Num features: 13
  Cat features: 37
  Validation  : PASS


In [39]:
# ==============================================================================
# 01.24 — SAVE VALIDATED DATASET REGISTRY
# ==============================================================================

print("=" * 100)
print("01.24 — SAVE VALIDATED DATASET REGISTRY")
print("=" * 100)

REGISTRY_DIR = NOTEBOOK_01_DIR / "registry"
REGISTRY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VALIDATED_REGISTRY_PATH = (
    REGISTRY_DIR
    / "validated_dataset_registry.json"
)

with open(
    VALIDATED_REGISTRY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        VALIDATED_DATASET_REGISTRY,
        f,
        indent=2
    )

print("✓ Validated dataset registry saved:")
print(f"  {VALIDATED_REGISTRY_PATH}")

01.24 — SAVE VALIDATED DATASET REGISTRY
✓ Validated dataset registry saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/registry/validated_dataset_registry.json


In [40]:
# ==============================================================================
# 01.25 — SAVE RAW FILE FINGERPRINT MANIFEST
# ==============================================================================

print("=" * 100)
print("01.25 — SAVE RAW FILE FINGERPRINT MANIFEST")
print("=" * 100)

RAW_FINGERPRINT_MANIFEST = {
    "project": "SPP-GAN Research Project",
    "notebook": "01",
    "purpose": "Raw dataset provenance and SHA-256 fingerprinting",
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "datasets": RAW_FILE_FINGERPRINTS,
}

RAW_FINGERPRINT_MANIFEST_PATH = (
    MANIFEST_DIR
    / "raw_file_fingerprint_manifest.json"
)

with open(
    RAW_FINGERPRINT_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        RAW_FINGERPRINT_MANIFEST,
        f,
        indent=2
    )

print("✓ Raw file fingerprint manifest saved:")
print(f"  {RAW_FINGERPRINT_MANIFEST_PATH}")

01.25 — SAVE RAW FILE FINGERPRINT MANIFEST
✓ Raw file fingerprint manifest saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/manifest/raw_file_fingerprint_manifest.json


In [41]:
# ==============================================================================
# 01.26 — NOTEBOOK 01 PROVENANCE MANIFEST
# ==============================================================================

print("=" * 100)
print("01.26 — NOTEBOOK 01 PROVENANCE MANIFEST")
print("=" * 100)

NOTEBOOK_01_MANIFEST = {
    "project": "SPP-GAN Research Project",

    "notebook": "01",

    "title": "Raw Dataset Loading & Validation",

    "purpose": (
        "Official raw-data acquisition, preservation, "
        "loading, and validation."
    ),

    "timestamp_utc": (
        datetime.now(timezone.utc).isoformat()
    ),

    "python_version": sys.version,

    "platform": platform.platform(),

    "project_root": str(PROJECT_ROOT),

    "datasets": DATASET_IDS,

    "target_columns": {
        dataset_id: str(
            TARGET_COLUMNS[dataset_id]
        )
        for dataset_id in DATASET_IDS
    },

    "raw_files": {
        dataset_id: str(
            RAW_OUTPUT_FILES[dataset_id]
        )
        for dataset_id in DATASET_IDS
    },

    "raw_file_sha256": {
        dataset_id: RAW_FILE_FINGERPRINTS[
            dataset_id
        ]["sha256"]
        for dataset_id in DATASET_IDS
    },

    "dataset_shapes": DATASET_SHAPES,

    "validation_status": {
        dataset_id: DATASET_VALIDATION_RESULTS[
            dataset_id
        ]["status"]
        for dataset_id in DATASET_IDS
    },

    "scientific_transformations_performed": [
        "Adult source-format label normalization only: "
        "removed trailing period from adult.test target labels"
    ],

    "scientific_transformations_not_performed": [
        "imputation",
        "encoding",
        "scaling",
        "normalization",
        "feature engineering",
        "feature selection",
        "duplicate removal",
        "identifier removal",
        "train_validation_test_split",
        "model_training",
        "synthetic_data_generation",
        "differential_privacy",
        "statistical_evaluation",
        "machine_learning_evaluation",
    ],

    "artifacts": {
        "validated_dataset_registry": str(
            VALIDATED_REGISTRY_PATH
        ),
        "raw_file_fingerprint_manifest": str(
            RAW_FINGERPRINT_MANIFEST_PATH
        ),
        "raw_dataset_download_manifest": str(
            acquisition_manifest_path
        ),
        "dataset_validation_summary": str(
            DATASET_VALIDATION_SUMMARY_PATH
        ),
        "raw_missingness_summary": str(
            RAW_MISSINGNESS_SUMMARY_PATH
        ),
        "structural_anomaly_summary": str(
            STRUCTURAL_ANOMALY_SUMMARY_PATH
        ),
    },
}

NOTEBOOK_01_MANIFEST_PATH = (
    MANIFEST_DIR
    / "notebook_01_manifest.json"
)

with open(
    NOTEBOOK_01_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        NOTEBOOK_01_MANIFEST,
        f,
        indent=2
    )

print("✓ Notebook 01 provenance manifest saved:")
print(f"  {NOTEBOOK_01_MANIFEST_PATH}")

01.26 — NOTEBOOK 01 PROVENANCE MANIFEST
✓ Notebook 01 provenance manifest saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01/manifest/notebook_01_manifest.json


In [42]:
# ==============================================================================
# 01.27 — FINAL INTEGRITY VERIFICATION
# ==============================================================================

print("=" * 100)
print("01.27 — FINAL INTEGRITY VERIFICATION")
print("=" * 100)

FINAL_CHECKS = {}

# ------------------------------------------------------------------------------
# Dataset files
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    path = RAW_OUTPUT_FILES[dataset_id]

    FINAL_CHECKS[
        f"{dataset_id}_raw_file_exists"
    ] = path.exists()

    FINAL_CHECKS[
        f"{dataset_id}_raw_file_nonempty"
    ] = (
        path.exists()
        and path.stat().st_size > 0
    )

# ------------------------------------------------------------------------------
# Validation
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    FINAL_CHECKS[
        f"{dataset_id}_validation_pass"
    ] = (
        DATASET_VALIDATION_RESULTS[
            dataset_id
        ]["status"] == "PASS"
    )

# ------------------------------------------------------------------------------
# Targets
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    FINAL_CHECKS[
        f"{dataset_id}_target_resolved"
    ] = (
        dataset_id in TARGET_COLUMNS
        and TARGET_COLUMNS[dataset_id] is not None
    )

# ------------------------------------------------------------------------------
# Artifact existence
# ------------------------------------------------------------------------------

REQUIRED_FINAL_ARTIFACTS = [
    acquisition_manifest_path,
    VALIDATED_REGISTRY_PATH,
    RAW_FINGERPRINT_MANIFEST_PATH,
    NOTEBOOK_01_MANIFEST_PATH,
    DATASET_VALIDATION_SUMMARY_PATH,
    RAW_MISSINGNESS_SUMMARY_PATH,
    STRUCTURAL_ANOMALY_SUMMARY_PATH,
]

for artifact in REQUIRED_FINAL_ARTIFACTS:

    FINAL_CHECKS[
        f"artifact_{Path(artifact).name}"
    ] = Path(artifact).exists()

# ------------------------------------------------------------------------------
# Final status
# ------------------------------------------------------------------------------

FAILED_FINAL_CHECKS = [
    name
    for name, passed in FINAL_CHECKS.items()
    if not passed
]

FINAL_STATUS = (
    "PASS"
    if len(FAILED_FINAL_CHECKS) == 0
    else "FAIL"
)

print()
print("FINAL CHECK RESULTS")
print("-" * 100)

for name, passed in FINAL_CHECKS.items():

    symbol = "✓" if passed else "✗"

    print(
        f"{symbol} {name}"
    )

print()
print(f"FINAL STATUS: {FINAL_STATUS}")

if FAILED_FINAL_CHECKS:

    print()
    print("FAILED CHECKS:")

    for check in FAILED_FINAL_CHECKS:
        print(f"  ✗ {check}")

    raise RuntimeError(
        "Notebook 01 final integrity verification FAILED."
    )

print()
print("✓ All Notebook 01 integrity checks passed.")

01.27 — FINAL INTEGRITY VERIFICATION

FINAL CHECK RESULTS
----------------------------------------------------------------------------------------------------
✓ adult_income_raw_file_exists
✓ adult_income_raw_file_nonempty
✓ bank_marketing_raw_file_exists
✓ bank_marketing_raw_file_nonempty
✓ diabetes_130us_raw_file_exists
✓ diabetes_130us_raw_file_nonempty
✓ adult_income_validation_pass
✓ bank_marketing_validation_pass
✓ diabetes_130us_validation_pass
✓ adult_income_target_resolved
✓ bank_marketing_target_resolved
✓ diabetes_130us_target_resolved
✓ artifact_raw_dataset_download_manifest.json
✓ artifact_validated_dataset_registry.json
✓ artifact_raw_file_fingerprint_manifest.json
✓ artifact_notebook_01_manifest.json
✓ artifact_dataset_validation_summary.csv
✓ artifact_raw_missingness_summary.csv
✓ artifact_structural_anomaly_summary.csv

FINAL STATUS: PASS

✓ All Notebook 01 integrity checks passed.


In [43]:
# ==============================================================================
# 01.28 — FINAL COMPLETION SUMMARY
# ==============================================================================

print("=" * 100)
print("SPP-GAN RESEARCH PROJECT")
print("NOTEBOOK 01 — COMPLETION SUMMARY")
print("=" * 100)

print()
print("Raw Datasets")
print("-" * 100)

for dataset_id in DATASET_IDS:

    record = VALIDATED_DATASET_REGISTRY[
        dataset_id
    ]

    print(
        f"{dataset_id:20s} | "
        f"Rows: {record['rows']:>8,} | "
        f"Columns: {record['columns']:>3} | "
        f"Target: {record['target_column']:<12} | "
        f"{record['validation_status']}"
    )

print()
print("Provenance")
print("-" * 100)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:20s} : "
        f"{RAW_FILE_FINGERPRINTS[dataset_id]['sha256']}"
    )

print()
print("Validation")
print("-" * 100)

print("  ✓ Raw files exist")
print("  ✓ Raw files are non-empty")
print("  ✓ Dataset structures validated")
print("  ✓ Target columns resolved")
print("  ✓ Missingness characterized")
print("  ✓ Duplicate records characterized")
print("  ✓ Constant features characterized")
print("  ✓ Identifier candidates characterized")
print("  ✓ Numeric/categorical inventories generated")
print("  ✓ SHA-256 fingerprints generated")
print("  ✓ Provenance manifest generated")
print("  ✓ Validated dataset registry generated")
print("  ✓ Final artifact integrity verified")

print()
print("Scientific transformations")
print("-" * 100)

print("  ✓ No imputation")
print("  ✓ No encoding")
print("  ✓ No scaling")
print("  ✓ No feature engineering")
print("  ✓ No feature removal")
print("  ✓ No duplicate removal")
print("  ✓ No train/test splitting")
print("  ✓ No model training")
print("  ✓ No synthetic-data generation")
print("  ✓ No differential privacy")

print()
print("=" * 100)
print("NOTEBOOK 01 STATUS: COMPLETE / PASS")
print("=" * 100)

print()
print("NEXT NOTEBOOK:")
print("02 — Preprocessing, Encoding & Data Splits")
print("=" * 100)

SPP-GAN RESEARCH PROJECT
NOTEBOOK 01 — COMPLETION SUMMARY

Raw Datasets
----------------------------------------------------------------------------------------------------
adult_income         | Rows:   48,842 | Columns:  15 | Target: income       | PASS
bank_marketing       | Rows:   45,211 | Columns:  17 | Target: y            | PASS
diabetes_130us       | Rows:  101,766 | Columns:  50 | Target: readmitted   | PASS

Provenance
----------------------------------------------------------------------------------------------------
adult_income         : 9479f8b76861e48c66836d97937f2c5a469581672e12c795d770b297817ae3a1
bank_marketing       : d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d
diabetes_130us       : 0689e7ec031237dc63031b938805c48377748761a3b26acab621567afa24df97

Validation
----------------------------------------------------------------------------------------------------
  ✓ Raw files exist
  ✓ Raw files are non-empty
  ✓ Dataset structures validated
  ✓ Tar